In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline


---
## Preprocess 5CO2 and Picarro reference data
This code can be used to process the data of the 5CO2 sensor and the picarro reference co2 data. The result are two .json files which cover the same time slice.

In [ ]:
#Load the data
reference_filename = 'data/Picarro_N5_DWD.feather'
filename = 'data/temp_data.parquet'
data = pd.read_parquet(filename)
data['timestamp'] = pd.to_datetime(data['ts'], unit='ms')
data = data.drop(columns=['ts']).set_index('timestamp').sort_index()
reference = pd.read_feather(reference_filename)
data.info() 
#Select the time range for analysis
start = '2026-02-20 00:00:00'
end = '2026-03-03 00:00:00'
reference = reference.loc[start:end]
data = data.loc[start:end]
data.info()
#drop unnecessary columns and resample reference data to 10 minutes
reference = reference[['CO2_dry', 'slope', 'intercept', 'CO2_dry_corrected']].reset_index()
data = data.drop(columns=['system_name']).reset_index()
reference = reference.rename(columns={'index': 'timestamp'})
data = data.rename(columns={'temperature [°C]': 'temperature', 'relative_humidity [%rH]': 'humidity', 'pressure [hPa]': 'pressure', 'CO2 (MUX 0)': 'co2_mux_0', 'CO2 (MUX 1)': 'co2_mux_1', 'CO2 (MUX 3)': 'co2_mux_3', 'CO2 (MUX 5)': 'co2_mux_5', 'CO2 (MUX 7)': 'co2_mux_7'})
#data = data.rename(columns={'datetime': 'timestamp'})
#Convert to JSON format
reference.to_json('data/reference_picarro.json', orient='records', lines=True)
data.to_json('data/example_data.json', orient='records', lines=True)